# 과제 1: PEFT(LoRA)를 돌려보고 비교표에서 위치 해석하기

202021023 · 3주차 실습 노트북(NSMC 감성분류, KoELECTRA)을 바탕으로 **전체 미세조정(기준선)**, **LoRA r=4**, **LoRA r=8**을 같은 조건에서 돌려 비교한다.

## 1. 가정한 예산

도내 소규모 조직을 이렇게 가정한다.

- 전용 GPU 서버가 없다. **Colab 무료 티어, T4 GPU 1장**으로 학습한다
- 학습 한 번은 **30분 이내**에 끝나야 한다

## 2. 비교 조건

아래 조건은 모든 실험에서 같다. 달라지는 것은 **학습 방식(전체 미세조정 / LoRA 랭크)**이고, 네 번째 실험만 **학습률 하나**를 더 바꿨다.

- 모델: `monologg/koelectra-base-v3-discriminator` (3주차 실습과 같음)
- 데이터: NSMC 앞 3,000건 → 학습 2,400 / 평가 600 (`seed=42`)
- 학습: 1에폭, 배치 16, 학습률 5e-5(Trainer 기본값), `seed=42`. 네 번째 줄만 학습률 5e-4
- LoRA 설정: 3주차 실습과 같게 `lora_alpha=8`, `lora_dropout=0.1`, 대상 `query`, `value`
- 시간: `trainer.train()` 벽시계 시간 / 메모리: 학습 중 GPU 최대 할당량(`torch.cuda.max_memory_allocated`)
- 환경: Colab에 깔려 있던 `torchao` 0.10이 최신 `peft`와 충돌해 LoRA를 붙일 때 ImportError가 났다. 이 과제는 torchao를 쓰지 않아서 설치 칸에서 지웠다

In [1]:
# 필요한 것 설치 (Colab에서 한 번만)
!pip -q install transformers datasets peft accelerate evaluate
!pip -q uninstall -y torchao  # Colab에 깔린 torchao 0.10이 최신 peft와 충돌해서 지운다 (이 과제는 torchao를 안 씀)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.0 MB/s eta 0:00:00


In [2]:
from datasets import load_dataset

# 3주차 실습과 같은 데이터: NSMC 원본 CSV 앞 3,000건
data_url = "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt"
dataset = load_dataset("csv", data_files=data_url, sep="\t", split="train[:3000]")
split = dataset.train_test_split(test_size=0.2, seed=42)
train_ds, eval_ds = split["train"], split["test"]
print("train:", len(train_ds), "eval:", len(eval_ds))

Generating train split: 0 examples [00:00, ? examples/s]

train: 2400 eval: 600


In [3]:
from transformers import AutoTokenizer

model_name = "monologg/koelectra-base-v3-discriminator"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch["document"], padding="max_length", truncation=True, max_length=128)

train_ds = train_ds.map(tokenize, batched=True, remove_columns=["document"]).rename_column("label", "labels")
eval_ds = eval_ds.map(tokenize, batched=True, remove_columns=["document"]).rename_column("label", "labels")
train_ds.set_format("torch")
eval_ds.set_format("torch")

config.json:   0%|          | 0.00/467 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/61.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/263k [00:00<?, ?B/s]

Map:   0%|          | 0/2400 [00:00<?, ? examples/s]

Map:   0%|          | 0/600 [00:00<?, ? examples/s]

In [4]:
import gc, time
import numpy as np
import torch
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments, set_seed
from peft import LoraConfig, get_peft_model

gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print("런타임:", gpu)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    return {"accuracy": (np.argmax(logits, axis=-1) == labels).mean()}

def run(name, r=None, lr=5e-5):
    # r=None 이면 전체 미세조정, 숫자면 그 랭크로 LoRA. lr 기본값 5e-5는 Trainer 기본 학습률과 같다
    set_seed(42)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    base_total = sum(p.numel() for p in model.parameters())
    if r is not None:
        model = get_peft_model(model, LoraConfig(r=r, lora_alpha=8, lora_dropout=0.1, bias="none",
                                                 task_type="SEQ_CLS", target_modules=["query", "value"]))
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

    args = TrainingArguments(output_dir=f"./results-{name}", num_train_epochs=1,
                             per_device_train_batch_size=16, per_device_eval_batch_size=16,
                             learning_rate=lr, logging_steps=50, save_strategy="no", report_to=[], seed=42)
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=eval_ds,
                      compute_metrics=compute_metrics)

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    start = time.time()
    trainer.train()
    train_sec = time.time() - start
    peak_gb = torch.cuda.max_memory_allocated() / 1024**3 if torch.cuda.is_available() else float("nan")
    acc = trainer.evaluate()["eval_accuracy"]

    del trainer, model
    gc.collect()
    return {"기법": name, "학습 파라미터 수": trainable, "전체 대비 비율": trainable / base_total * 100,
            "학습 시간(초)": train_sec, "평가 정확도": acc, "메모리(GB)": peak_gb}

런타임: Tesla T4


In [5]:
results = [
    run("전체 미세조정"),
    run("LoRA r=4", r=4),
    run("LoRA r=8", r=8),
    run("LoRA r=8, 학습률 5e-4", r=8, lr=5e-4),   # 위 줄에서 학습률 하나만 바꿨다
]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  452MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraForSequenceClassification LOAD REPORT from: monologg/koelectra-base-v3-discriminator
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 
classifier.dense.bias                             | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your do

model.safetensors: reconstructing file:   0%|          |  0.00B /  452MB            

...

Training Loss,Validation Loss,Step,Accuracy
0.690365,0.686239,150,0.578333


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraForSequenceClassification LOAD REPORT from: monologg/koelectra-base-v3-discriminator
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 
classifier.dense.bias                             | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your do

Step,Training Loss
50,0.703051
100,0.619278
150,0.497996


Training Loss,Validation Loss,Step,Accuracy
0.497996,0.434970,150,0.796667


In [6]:
# 비교표 (README 양식)
print(f"| 기법 | 학습 파라미터 수 | 전체 대비 비율 | GPU/런타임 | 학습 시간 | 평가 정확도 | 메모리 사용량 |")
print(f"|---|---|---|---|---|---|---|")
for x in results:
    print(f"| {x['기법']} | {x['학습 파라미터 수']:,} | {x['전체 대비 비율']:.2f}% | {gpu} | "
          f"{x['학습 시간(초)']:.1f}초 | {x['평가 정확도']:.3f} | {x['메모리(GB)']:.2f} GB |")

| 기법 | 학습 파라미터 수 | 전체 대비 비율 | GPU/런타임 | 학습 시간 | 평가 정확도 | 메모리 사용량 |
|---|---|---|---|---|---|---|
| 전체 미세조정 | 112,922,882 | 100.00% | Tesla T4 | 46.9초 | 0.842 | 2.51 GB |
| LoRA r=4 | 739,586 | 0.65% | Tesla T4 | 39.3초 | 0.578 | 1.44 GB |
| LoRA r=8 | 887,042 | 0.79% | Tesla T4 | 37.9초 | 0.578 | 1.45 GB |
| LoRA r=8, 학습률 5e-4 | 887,042 | 0.79% | Tesla T4 | 37.6초 | 0.797 | 1.45 GB |


## 3. 비교표와 해석

| 기법 | 학습 파라미터 수 | 전체 대비 비율 | GPU/런타임 | 학습 시간 | 평가 정확도 | 메모리 사용량 |
|---|---|---|---|---|---|---|
| 전체 미세조정 (기준선) | 112,922,882 | 100% | Colab T4 | 46.9초 | 0.842 | 2.51 GB |
| LoRA r=4 | 739,586 | 0.65% | Colab T4 | 39.3초 | 0.578 | 1.44 GB |
| LoRA r=8 | 887,042 | 0.79% | Colab T4 | 37.9초 | 0.578 | 1.45 GB |
| LoRA r=8, 학습률 5e-4 | 887,042 | 0.79% | Colab T4 | 37.6초 | 0.797 | 1.45 GB |

- **학습 파라미터**: LoRA r=4는 전체의 0.65%만 학습한다. r을 4에서 8로 두 배 하면 LoRA 행렬 몫이 147,456개에서 294,912개로 정확히 두 배가 된다. 분류 헤드 592,130개는 그대로라 전체로는 두 배가 아니다
- **메모리**: 2.51GB에서 1.44GB로 약 43% 줄었다. 학습하는 파라미터가 적어 기울기와 옵티마이저 상태가 작아진 몫이다
- **시간**: 46.9초에서 37.6~39.3초로 16~20%밖에 줄지 않았다. 얼려 둔 본체도 순전파와 역전파를 거쳐야 해서 계산량 자체는 크게 줄지 않는다. 시간은 실행마다 몇 초씩 달랐다(첫 실행에서는 51.8 / 40.5 / 41.8초)
- **정확도**: 같은 학습률(5e-5)에서 LoRA는 0.578로 기준선 0.842에 크게 못 미쳤고, r=4와 r=8이 같았다. 랭크가 병목이 아니라는 뜻이다. 학습률만 5e-4로 올리자 0.797이 되어 기준선과의 차이가 0.045로 줄었다. 앞의 낮은 점수는 LoRA 자체의 한계라기보다 학습률이 작아 1에폭 동안 덜 배운 탓으로 본다
- **한계**: 1에폭, 2,400건, 시드 하나로 한 번씩만 돌렸다. 정확도는 두 번 돌려도 같았다. 학습률은 5e-5와 5e-4 두 값만 봤고, r=4에 학습률 5e-4는 돌려보지 않았다

## 4. 선택한 기법과 근거

**LoRA r=8, 학습률 5e-4**를 고른다.

- **예산 안에 든다**: T4 한 장에서 학습 38초, GPU 메모리 1.45GB다. 30분 예산에 한참 못 미친다
- **왜 전체 미세조정이 아니라 PEFT인가**: 이 규모에서는 전체 미세조정도 예산 안에 들어가고 정확도가 0.045 높다. 그래도 LoRA를 고른 이유는 두 가지다
  1. 메모리를 43% 덜 써서, 같은 T4에서 더 큰 모델이나 더 큰 배치로 넘어갈 여유가 생긴다
  2. 학습 결과로 남길 것이 1.1억 개 전체가 아니라 88만 개(0.79%)다. fp32 기준으로 계산하면 약 450MB 대신 약 3.5MB다. 문의 분류, 감성 분석처럼 과제가 늘어도 본체 하나에 작은 어댑터만 더하면 되니, 저장과 관리 비용이 작다
- **감수한 것**: 기준선보다 정확도가 0.045 낮다. 에폭이나 학습률을 더 조정하면 이 차이가 줄어드는지는 확인하지 않았다
- **r=8인 이유**: 학습률 5e-4로 확인한 것이 r=8이다. 학습률 5e-5에서는 r=4와 r=8의 정확도·메모리·시간이 같았으니 r=4도 비슷할 수 있지만, 확인은 하지 않았다